We can count the number of parameters in a neural network empirically with this function.

In [ ]:
def count_params(model):
    breakdown = {}
    for name, module in model.named_children():
        n = sum(p.numel() for p in module.parameters())
        breakdown[name] = n
    total = sum(breakdown.values())
    return breakdown, total

In [ ]:
import mini_gpt

config = mini_gpt.ModelConfig()  # uses all defaults
print(config)

model = mini_gpt.MiniTransformer(config)
breakdown, total = count_params(model)

print("\nParameter breakdown:")
for name, n in breakdown.items():
    print(f"  {name}: {n:,} ({n / 1e3:.1f}K)")
print(f"\nTotal: {total:,} ({total / 1e6:.2f}M)")

ModelConfig(vocab_size=1000, hidden_dim=256, num_layers=4, num_heads=4, ff_dim=1024)

Parameter breakdown:
  embedding: 256,000 (256.0K)
  layers: 3,159,040 (3159.0K)
  final_norm: 512 (0.5K)
  lm_head: 256,000 (256.0K)

Total: 3,671,552 (3.67M)


Remembering that each parameter is a number, which we need to store in fp32, is 4 bytes.

- $m_{\text{params}} = 4 \times N$
- $m_{\text{grad}} = 4 \times N$
- $m_{\text{opt}} = (4 + 4) \times N$

In [11]:
bytes_per_param = 4 + 4 + 8  # param + grad + optimizer states
train_memory = total * bytes_per_param

print(f"Params: {total * 4 / 1e6:.2f} MB")
print(f"Grads:  {total * 4 / 1e6:.2f} MB")
print(f"Optim:  {total * 8 / 1e6:.2f} MB")
print(f"Total training memory: {train_memory / 1e6:.2f} MB")

Params: 10.48 MB
Grads:  10.48 MB
Optim:  20.97 MB
Total training memory: 41.93 MB
